# Score calculations for all six embeddings

## LogME score

In [2]:
import os
import json
import numpy as np

In [5]:
def compute_logme(features: np.ndarray, labels: np.ndarray, max_iter: int = 25, tol: float = 1e-4):
    """
    Computes overall and per-class LogME scores with stable MacKay fixed-point updates.
    """
    N, D = features.shape
    unique_classes = np.unique(labels)
    num_classes = len(unique_classes)
    
    # 1. Mean-centre features
    F = features - np.mean(features, axis=0, keepdims=True)
    
    # 2. Thin SVD: F = U * diag(s) * V^T
    U, s, _ = np.linalg.svd(F, full_matrices=False)
    k = len(s)
    s2 = s ** 2

    class_scores = {}
    total_log_evidence = 0.0

    for c in unique_classes:
        # Binary target centred to remove DC bias
        y = (labels == c).astype(np.float64)
        y_centered = y - np.mean(y)
        
        y_proj = U.T @ y_centered          # Shape: (k,)
        y_norm_sq = np.dot(y_centered, y_centered)
        y_proj_sq = np.sum(y_proj ** 2)
        y_ortho_sq = max(0.0, y_norm_sq - y_proj_sq)  # Energy outside feature span

        # Initialise hyperparameters
        alpha = 1.0
        beta = 1.0 / (np.var(y_centered) + 1e-4)

        for _ in range(max_iter):
            prev_alpha, prev_beta = alpha, beta
            
            denom = alpha + beta * s2
            gamma = np.sum((beta * s2) / denom)
            
            m = (beta * s / denom) * y_proj
            w_norm_sq = np.sum(m ** 2)
            
            rss = y_ortho_sq + np.sum(((alpha / denom) * y_proj) ** 2)

            alpha = gamma / (w_norm_sq + 1e-12)
            beta = max(1e-12, N - gamma) / (rss + 1e-12)

            if abs(alpha - prev_alpha) / (alpha + 1e-12) < tol and abs(beta - prev_beta) / (beta + 1e-12) < tol:
                break

        # Log determinant: log |C|
        log_det = -N * np.log(beta + 1e-12) - k * np.log(alpha + 1e-12) + np.sum(np.log(alpha + beta * s2 + 1e-12))
        
        # Quadratic form: y^T C^{-1} y
        quad_form = beta * y_ortho_sq + np.sum((alpha * beta / (alpha + beta * s2 + 1e-12)) * (y_proj ** 2))
        
        # Marginal log likelihood for class c
        log_evidence_c = -0.5 * (N * np.log(2.0 * np.pi) + log_det + quad_form)
        
        class_score = float(log_evidence_c / N)
        class_scores[int(c)] = class_score
        total_log_evidence += log_evidence_c

    overall_score = float(total_log_evidence / (N * num_classes))
    return overall_score, class_scores


def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []

    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}.")
        
        # Take the first k sample indices deterministically
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


# --- Configuration ---
SHOTS_PER_CLASS = 1   # Set to 10, 5, or 1

base_dir = "./model_embeddings"
output_dir = f"./model_scores_{SHOTS_PER_CLASS}_shot/logME"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Evaluating {model_name} on first {SHOTS_PER_CLASS} samples/class...")
    data = np.load(npz_path)
    full_features = data["embeddings"]
    full_labels = data["labels"]
    
    # Deterministic subset extraction
    features, labels = get_first_k_subset(
        full_features, 
        full_labels, 
        shots_per_class=SHOTS_PER_CLASS
    )
    
    overall, class_wise = compute_logme(features, labels)
    
    results[model_name] = {
        "shots_per_class": SHOTS_PER_CLASS,
        "total_samples": int(features.shape[0]),
        "overall_logme": overall,
        "class_wise_logme": class_wise
    }

output_file = os.path.join(output_dir, "all_models_logme.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved deterministic {SHOTS_PER_CLASS}-shot LogME scores to {output_file}")

Evaluating AVES-bio on first 1 samples/class...
Evaluating BirdAVES-biox-base on first 1 samples/class...
Evaluating DistilHuBERT on first 1 samples/class...
Evaluating HuBERT on first 1 samples/class...
Evaluating MusicHuBERT on first 1 samples/class...
Evaluating WavLM on first 1 samples/class...

Saved deterministic 1-shot LogME scores to ./model_scores_1_shot/logME\all_models_logme.json


## H score

In [8]:
import os
import json
import numpy as np

def compute_hscore(features: np.ndarray, labels: np.ndarray, eps: float = 1e-4):
    """
    Computes overall and per-class H-scores for features (N, D) and labels (N,).
    """
    N, D = features.shape
    unique_classes, counts = np.unique(labels, return_counts=True)
    
    # 1. Mean-centre features across samples
    F_centered = features - np.mean(features, axis=0, keepdims=True)

    # 2. Total feature covariance matrix with Tikhonov regularisation
    Sigma_f = (F_centered.T @ F_centered) / N
    Sigma_f_reg = Sigma_f + eps * np.eye(D)
    Sigma_f_inv = np.linalg.pinv(Sigma_f_reg)

    # 3. Class-wise centroid Mahalanobis evaluation
    class_scores = {}
    overall_hscore = 0.0

    for c, count in zip(unique_classes, counts):
        idx = np.where(labels == c)[0]
        mu_c = np.mean(F_centered[idx], axis=0)  # Class-conditional mean vector (D,)
        
        # Squared Mahalanobis distance of class centroid
        h_c = float(mu_c.T @ Sigma_f_inv @ mu_c)
        class_scores[int(c)] = h_c
        
        # Prior-weighted contribution to the overall trace
        overall_hscore += (count / N) * h_c

    return float(overall_hscore), class_scores


def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []

    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}.")
        
        # Take the first k sample indices deterministically
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


# --- Configuration ---
SHOTS_PER_CLASS = 1   # Set to 10, 5, or 1

base_dir = "./model_embeddings"
output_dir = f"./model_scores_{SHOTS_PER_CLASS}_shot/H_score"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Evaluating H-score for {model_name} on first {SHOTS_PER_CLASS} samples/class...")
    data = np.load(npz_path)
    full_features = data["embeddings"]
    full_labels = data["labels"]
    
    # Deterministic subset extraction
    features, labels = get_first_k_subset(
        full_features, 
        full_labels, 
        shots_per_class=SHOTS_PER_CLASS
    )
    
    overall, class_wise = compute_hscore(features, labels)
    
    results[model_name] = {
        "shots_per_class": SHOTS_PER_CLASS,
        "total_samples": int(features.shape[0]),
        "overall_hscore": overall,
        "class_wise_hscore": class_wise
    }

output_file = os.path.join(output_dir, "all_models_hscore.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved deterministic {SHOTS_PER_CLASS}-shot H-scores to {output_file}")

Evaluating H-score for AVES-bio on first 1 samples/class...
Evaluating H-score for BirdAVES-biox-base on first 1 samples/class...
Evaluating H-score for DistilHuBERT on first 1 samples/class...
Evaluating H-score for HuBERT on first 1 samples/class...
Evaluating H-score for MusicHuBERT on first 1 samples/class...
Evaluating H-score for WavLM on first 1 samples/class...

Saved deterministic 1-shot H-scores to ./model_scores_1_shot/H_score\all_models_hscore.json


# NTK - LFC

In [15]:
import os
import json
import numpy as np

def compute_ntk_lfc(features: np.ndarray, labels: np.ndarray, centered: bool = True):
    """
    Computes overall and class-wise NTK LFC with deterministic, consistent decomposition.
    """
    N, D = features.shape
    unique_classes, counts = np.unique(labels, return_counts=True)
    num_classes = len(unique_classes)
    
    # 1. Unit-norm normalise embeddings
    norms = np.linalg.norm(features, axis=1, keepdims=True) + 1e-8
    F_norm = features / norms

    # 2. Linear NTK / Gram Matrix
    K = np.clip(F_norm @ F_norm.T, -1.0, 1.0)
    
    # Construct ideal block-diagonal label kernel K_Y
    Y_onehot = np.eye(num_classes)[labels]  # (N, C)
    K_Y = Y_onehot @ Y_onehot.T             # (N, N)

    # 3. Kernel Centering: H * K * H, where H = I - (1/N) 1 1^T
    if centered:
        H = np.eye(N) - (1.0 / N) * np.ones((N, N))
        K = H @ K @ H
        K_Y = H @ K_Y @ H

    norm_K = np.linalg.norm(K, 'fro') + 1e-12
    norm_KY = np.linalg.norm(K_Y, 'fro') + 1e-12

    # 4. Class-wise decomposition
    class_scores = {}
    total_alignment = 0.0

    for c in unique_classes:
        y_c = (labels == c).astype(np.float64)
        
        if centered:
            # Centred target vector
            y_c_tilde = y_c - np.mean(y_c)
            intra_c = float(y_c_tilde.T @ K @ y_c_tilde)
        else:
            idx = np.where(labels == c)[0]
            intra_c = float(np.sum(K[np.ix_(idx, idx)]))

        score_c = intra_c / (norm_K * norm_KY)
        class_scores[int(c)] = score_c
        total_alignment += score_c

    overall_lfc = float(total_alignment)
    return overall_lfc, class_scores


def get_first_k_subset(features: np.ndarray, labels: np.ndarray, shots_per_class: int):
    """
    Deterministically selects the first `shots_per_class` samples for each class
    in the order they appear in the archive.
    """
    selected_indices = []

    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) < shots_per_class:
            raise ValueError(f"Class {c} only has {len(c_indices)} samples; cannot pick {shots_per_class}.")
        
        selected_indices.extend(c_indices[:shots_per_class])

    selected_indices = np.array(selected_indices)
    return features[selected_indices], labels[selected_indices]


# --- Configuration ---
SHOTS_PER_CLASS = 1   # Set to 10, 5, or 1

base_dir = "./model_embeddings"
output_dir = f"./model_scores_{SHOTS_PER_CLASS}_shot/LFC_score"
os.makedirs(output_dir, exist_ok=True)

results = {}

for model_name in sorted(os.listdir(base_dir)):
    model_path = os.path.join(base_dir, model_name)
    if not os.path.isdir(model_path):
        continue
        
    npz_path = os.path.join(model_path, "features.npz")
    if not os.path.exists(npz_path):
        continue
        
    print(f"Evaluating LFC for {model_name} on first {SHOTS_PER_CLASS} samples/class...")
    data = np.load(npz_path)
    full_features = data["embeddings"]
    full_labels = data["labels"]
    
    # Deterministic subset extraction
    features, labels = get_first_k_subset(
        full_features, 
        full_labels, 
        shots_per_class=SHOTS_PER_CLASS
    )
    
    overall, class_wise = compute_ntk_lfc(features, labels, centered=True)
    
    results[model_name] = {
        "shots_per_class": SHOTS_PER_CLASS,
        "total_samples": int(features.shape[0]),
        "overall_lfc": overall,
        "class_wise_lfc": class_wise
    }

output_file = os.path.join(output_dir, "all_models_lfc.json")
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"\nSaved deterministic {SHOTS_PER_CLASS}-shot centered LFC scores to {output_file}")

Evaluating LFC for AVES-bio on first 1 samples/class...
Evaluating LFC for BirdAVES-biox-base on first 1 samples/class...
Evaluating LFC for DistilHuBERT on first 1 samples/class...
Evaluating LFC for HuBERT on first 1 samples/class...
Evaluating LFC for MusicHuBERT on first 1 samples/class...
Evaluating LFC for WavLM on first 1 samples/class...

Saved deterministic 1-shot centered LFC scores to ./model_scores_1_shot/LFC_score\all_models_lfc.json
